## Overview

The Burgers' equation is a paradigmatic nonlinear differential equation defined in 1-D as 
$$
\frac{\partial u}{\partial t} = \nu \frac{\partial^2 u}{\partial x^2} - u \frac{\partial u}{\partial x} ,\quad u(x,0) = u_0(x),
$$
where $u(x,t)$ is the fluid velocity at poisition $x$ and time $t$, $\nu$ is the diffusion coefficient and $u_0(x)$ is the initial condition. 

Since the Burgers' equation is nonlinear, we embed it into a higher order linear system using the following transformations:
- Spatial discretization to transform the nonlinear PDE $\rightarrow$ nonlinear ODE.
- Carleman linearization to transform the finite, nonlinear ODE $\rightarrow$ infinite, linear ODE.
- Temporal discretization & truncation to transform the infinite, linear ODE $\rightarrow$ finite linear system of equations.

The result of this is the following linear system $L^{(\text{e})}Y^{(\text{e})}=B^{(\text{e})}$ (see DH2026 for details). This linear system is solved using the following methods:
- An LCNU of the Carleman linear Burgers' equation is used to efficiently load the linear system onto quantum hardware.
- An initial condition is loaded using a state preparation routine.
- The Variational Quantum Linear Solver (VQLS) solves the linear system.

## Libraries

This has been tested using the following versions:
- python: 3.14.3
- numpy: 2.52
- matplotlib: 3.11.1
- qiskit: 2.52
- qiskit_aer: 0.17.2
- qiskit_algorithms: 0.4.0

In [1]:
#Libraries
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile, qpy
from qiskit.circuit.library import StatePreparation
from qiskit.quantum_info import Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_algorithms.optimizers import GradientDescent, CG, L_BFGS_B
import time

#Custom libs
import Burgers_utils as bu
import circuit_utils as cu
import misc_utils as mu
import Carleman_Dilation


## Parameters

Define parameters. 

In [2]:
#Parameters (each must be power of 2)
nx = 2**2 #number of spatial discretization points (>= 2**2)
nt = 2**1 #number of temporal discretization points
alpha = 2**1 #Carleman truncation order. Note: Carleman matrix scales exponentially with alpha

#Spatial parameters & initial condition
Length = 2*np.pi #units: [m]
dx   = Length/(nx-1)
ic_type = 'Gaussian' #'Gaussian','Impulse

#Temporal parameters
T = 5.       #Total duration, units: [s]
dt = T/nt

#Reynolds_Number. Use this to calculate nu
rey = 5
#nu = np.average(b_init)*Length/rey #using mean velocity
nu = 0.25  #0.25 #1. #5*10**-2  #[m^2/s]

#Total number of qubits needed for encoding +1 for zero padding and +1 is for ancilla
nqubit = int(np.log2(nt) + alpha*np.log2(nx) + 1 + 1)

#Ansatz Parameters
lAnsatz = 'Sim18' #Sim9Mod','Sim18'
nlayer = 4
if (lAnsatz=='Sim9Mod'):
    ntheta = nlayer * (nqubit-1)
elif (lAnsatz=='Sim18'):
    #ntheta = 3 * nlayer * (nqubit-1)
    ntheta = 2 * nlayer * (nqubit-1)
thetas = np.random.uniform(0,np.pi,ntheta)

#Optimizer parameters
opt = 'L_BFGS_B' #'GradientDescent', 'CG', 'L_BFGS_B'

#Set of parameters 
params = {'alpha':alpha,'nx':nx,'nt':nt,'nqubit':nqubit,
          'dt':dt,'dx':dx,'nu':nu,'nlayer':nlayer,
          'ntheta':ntheta,'Length':Length,'ic_type':ic_type,
          'opt':opt}

#Options
lValidate = False #Validate the circuits for generating the Carlman matrix
lPauliCompare = False #Compare the cost of our approach with the Pauli Decomposition
lCreateCircs = False #Create and transpile circuits from scratch (slower), otherwise load from file (faster)

print('Number of spatial grid points: nx = ',nx)
print('Number of temporal grid points: nt = ',nt)
print('Carleman truncation order: α = ', alpha)
print('Size of linear system: ', 2**(nqubit-1))
print('Number of ansatz layers: nlayer = ',nlayer)
print('Number of variational parameters: nθ = ',ntheta)
print('Initial condition type: ic_type = ',ic_type)
print('optimizer: opt = ',opt)

Number of spatial grid points: nx =  4
Number of temporal grid points: nt =  2
Carleman truncation order: α =  2
Size of linear system:  64
Number of ansatz layers: nlayer =  4
Number of variational parameters: nθ =  48
Initial condition type: ic_type =  Gaussian
optimizer: opt =  L_BFGS_B


## Create Circuits

Create the following:
- LCNU circuits (and coefficients) for $L^{(\text{e})}$, 
- ansatz circuit,
- initial condition circuit.

The number of terms in the LCNU of $L^{(\text{e})}$ is exactly $N_s=\frac{1}{2}(9\alpha^2 + \alpha + 8)$.
For $\alpha=2$ this gives $N_s=23$.

If `lValidate=True`, then we validate $L^{(\text{e})}$ from the circuits against its analytical form.


In [ ]:
#Create the circuits and coefficients for the Carleman matrix
qc_carl,coeffs = bu.create_circs(params)
Ns = len(coeffs) #number of circuits
params['Ns'] = Ns
# #Approximate the matrix based on the coefficients
# coeffs = [x for x in coeffs if x>coeff_thresh]
# qc_carl = [qc_carl[i] for i,x in enumerate(coeffs) if x>coeff_thresh]

#Validate the circuits and coefficients by comparing the Carleman matrix generated from the circuits 
#with the Carleman matrix generated from matrices
if (nqubit <= 11 and lValidate):
    Le_real,Le_circ = bu.validate_CarlemanDilated_Matrix(qc_carl,coeffs,params)

    #Plot exact approach versus circuit approach
    fig, ax = plt.subplots(1,3)
    p1 = ax[0].spy(Le_real, aspect = 'auto', markersize=2, alpha = 1.)
    p2 = ax[1].spy(Le_circ, aspect = 'auto', markersize=2, alpha = 1.)
    p3 = ax[2].spy(Le_real-Le_circ, aspect = 'auto', markersize=2, alpha = 1.)
    for axi in ax:
        axi.set_aspect('equal')
    plt.savefig("TestMatrix.png")

#Create Ansatz  
if (lAnsatz=='Sim9Mod'):
    qc_anz = cu.Ansatz_Sim9_Modified(params)
elif (lAnsatz=='Sim18'):
    qc_anz = cu.Ansatz_Sim18(params)

#Initial condition circuit
Be = mu.normalized_initial_condition(params) #normalized vector
Be = np.asarray(Be, dtype=complex)
stateprep = StatePreparation(Be)
qc_init0 = QuantumCircuit(nqubit-1)
qc_init0.append(stateprep, list(range(0,nqubit-1)))
#qc_init = qc_init0.decompose().decompose().decompose().decompose().decompose()
qc_init = qc_init0.decompose(reps=5)

#Transpilation 
simulator = AerSimulator()
qc_anz = transpile(qc_anz, simulator, optimization_level=3)

## Store observable in cache

Each interation of VQLS requires $N_s^2(q+1)$ circuits. Since this is slow to run, we will store the fixed observable for each iteration in cache.

In [ ]:
#U_b Z_k U_b^dagger
UZUd = bu.extract_UZUd(qc_init,params)

#Store A_j^dagger U_bZ_kU_b^dagger A_i in cache
Aj = bu.extract_Aj_from_Uj(qc_carl,params)

obs_beta = 0
obs_delta = 0
coeffs_ij = []
coeffs_ijk = []
for i in range(0,Ns):
    for j in range(0,Ns):
        c = coeffs[i] * np.conjugate(coeffs[j])
        coeffs_ij.append(c) #ij
        obs_beta += c * Aj[j].conj().T @ Aj[i]
        for k in range(0,nqubit-1):
            coeffs_ijk.append(c) #ijk
            obs_delta += c * Aj[j].conj().T @ UZUd[k] @ Aj[i]
del([c])

# #Plot spectrum
# fig, ax = plt.subplots(1,2)
# p1 = ax[0].plot(sorted(np.abs(coeffs)))
# p2 = ax[1].plot(sorted(np.abs(coeffs_ij)))
# ax[0].set_yscale('log') 
# ax[1].set_yscale('log') 
# plt.savefig("Coefficients.png")

In [ ]:
#Calculate cost function using cached matrices

# #N=Sim18Modified, nlayer=20, Impulse, nx=8, nt=2
# thetas = [1.51400429e+00, 3.20241083e+00, 1.00119998e+00, 1.52841552e+00,
#        5.97608413e-01, 1.43018364e+00, 2.70391830e+00, 6.94347029e-02,
#        2.11346890e+00, 2.28010667e+00, 1.19661184e+00, 1.48644579e+00,
#        2.42612683e+00, 2.82166201e+00, 1.35114623e+00, 1.47123599e+00,
#        2.57742400e+00, 5.37951878e-01, 1.51848700e+00, 5.13821718e-01,
#        1.52891862e+00, 1.87779722e+00, 6.90424245e-01, 1.80040693e+00,
#        2.04393412e+00, 1.43801853e+00, 1.16824267e+00, 6.75137998e-01,
#        5.03842697e-03, 1.67949917e+00, 2.11059179e+00, 3.36729927e+00,
#        1.32578540e+00, 1.65179345e+00, 1.45150668e+00, 7.15822638e-01,
#        1.44271763e+00, 2.80947486e-01, 2.36386815e+00, 8.16588079e-01,
#        1.88934278e+00, 1.07414671e+00, 1.49411225e+00, 3.05335223e+00,
#        1.00258733e+00, 1.39017135e+00, 7.84531909e-01, 1.90584153e+00,
#        0.00000000e+00, 5.00225600e-02, 5.74174335e-01, 1.21005901e+00,
#        7.31856118e-01, 1.55172379e+00, 7.65197178e-01, 1.87192235e+00,
#        3.46175893e+00, 2.87132244e+00, 1.21319843e+00, 3.77628311e+00,
#        2.22445092e-03, 1.82223321e+00, 1.92525552e+00, 1.97694399e+00,
#        9.13872371e-01, 3.01831138e+00, 1.00466040e+00, 7.46961878e-01,
#        1.77383602e+00, 2.56663975e+00, 3.35867339e-01, 2.13350608e+00,
#        2.76621132e+00, 1.46259303e+00, 2.37811333e+00, 2.58993723e+00,
#        3.02449022e+00, 2.52588984e+00, 8.60977448e-02, 3.47056453e+00,
#        1.03866211e+00, 1.80034949e+00, 9.89790432e-01, 1.49601823e+00,
#        1.15098251e+00, 1.11832149e+00, 2.54447116e+00, 2.13398977e+00,
#        1.39765971e+00, 3.04706660e+00, 3.02641462e+00, 1.27265588e+00,
#        2.39867088e+00, 8.13867291e-01, 1.36420952e+00, 2.19626807e+00,
#        2.89626000e+00, 2.10047577e+00, 2.88469383e+00, 1.96621988e+00,
#        2.06632340e+00, 2.61816369e+00, 5.13923971e-02, 6.39601083e-01,
#        2.71161068e+00, 7.16201665e-01, 1.68143640e+00, 1.06819749e+00,
#        3.04234331e+00, 1.72256803e+00, 2.58712416e+00, 1.50840685e+00,
#        2.97864928e+00, 4.51260401e-01, 8.74200686e-01, 2.08244167e+00,
#        1.52114783e-04, 1.64043255e+00, 1.83094434e+00, 2.16784280e+00,
#        1.24472269e+00, 1.61260434e+00, 5.76405702e-01, 5.65382708e-01,
#        3.29845574e+00, 3.27755010e+00, 9.03802877e-01, 1.92207521e+00,
#        0.00000000e+00, 2.28068668e-01, 1.72590187e+00, 1.77887140e-01,
#        2.26452665e+00, 0.00000000e+00, 1.68837570e+00, 7.95688924e-01,
#        6.21119841e-01, 9.46213735e-01, 2.78106368e+00, 3.27850949e+00,
#        0.00000000e+00, 8.70304020e-01, 1.67420405e+00, 2.13337169e+00,
#        3.31188496e+00, 1.00399285e+00, 2.83409041e+00, 9.83530549e-01,
#        9.02324545e-01, 1.22825637e+00, 2.02962259e+00, 3.55331389e+00,
#        9.04207483e-01, 1.51416927e+00, 1.26548764e-01, 4.56559292e-01,
#        2.85778128e+00, 8.41815043e-01, 5.85443328e-01, 1.19320293e+00,
#        0.00000000e+00, 1.82709192e+00, 2.76533071e+00, 2.46716063e+00,
#        1.66024391e+00, 1.29654041e+00, 1.04638082e-03, 9.08340917e-01,
#        1.65330387e+00, 2.10879945e+00, 1.79461048e+00, 2.96107835e+00,
#        4.75661384e-01, 1.40818105e+00, 1.92223840e+00, 1.03485787e+00,
#        1.40147694e+00, 2.62221273e-02, 1.87989184e+00, 3.25903571e+00,
#        1.97789911e+00, 2.71601670e+00, 6.67757704e-01, 1.27103955e+00,
#        7.02460933e-01, 6.47888646e-01, 8.47733692e-02, 6.30874784e-01,
#        2.76876755e+00, 1.61444628e+00, 1.59180274e+00, 2.50929043e-01,
#        1.15753282e+00, 1.75290524e+00, 1.70964044e+00, 1.99648501e+00,
#        2.04259798e+00, 1.94860783e+00, 9.27612084e-01, 2.44126553e+00,
#        2.48807793e-01, 2.84025849e+00, 2.90772128e+00, 2.79504424e+00,
#        2.59055836e+00, 4.33165624e-01, 1.52719794e+00, 5.75383720e-02,
#        3.08329460e+00, 1.17479591e+00, 1.89051213e+00, 2.61892431e+00,
#        2.72577860e+00, 1.51159488e+00, 3.33569884e-01, 8.96997973e-01,
#        1.81061439e+00, 1.15951441e-01, 1.17710435e+00, 2.12510706e-01,
#        3.40415840e+00, 1.98391030e+00, 1.82980584e+00, 4.05554145e+00,
#        1.04042410e+00, 1.61626138e+00, 2.23568205e+00, 1.98487812e-01,
#        1.43963540e-01, 1.85899663e+00, 3.25212708e+00, 3.54556190e-04,
#        6.65583867e-01, 4.56671212e-01, 3.95068237e+00, 2.28429744e+00,
#        1.98544760e+00, 8.12251610e-01, 8.34844930e-04, 6.71087318e-01,
#        3.87330244e-01, 1.42583072e+00, 1.71997763e+00, 2.59804040e+00,
#        2.98757417e+00, 1.14569540e+00, 3.67299545e-02, 0.00000000e+00,
#        1.19279231e+00, 1.56090343e+00, 3.73712251e+00, 2.19616506e+00,
#        2.06376246e+00, 7.57246878e-01, 3.24890476e+00, 6.09219052e-01,
#        9.70525567e-01, 4.75273231e-01, 2.07204554e+00, 1.20299322e+00,
#        1.42841651e+00, 1.47760034e+00, 1.98745646e+00, 9.07108902e-01,
#        2.33105987e+00, 2.12553976e+00, 1.31501406e+00, 3.20913356e+00,
#        6.78807046e-01, 1.99736561e+00, 1.92582371e+00, 8.22704181e-02,
#        1.79977379e-01, 3.09492324e+00, 3.01853548e+00, 1.93972944e+00,
#        1.53855031e+00, 3.59225799e+00, 1.38927349e+00, 8.69188078e-01,
#        7.79151837e-01, 2.10014724e+00, 2.08031271e+00, 1.67607287e+00,
#        1.97945702e+00, 3.09148308e+00, 1.51674354e+00, 1.12340891e+00,
#        6.48662408e-01, 1.98565003e+00, 2.13671009e+00, 2.46563023e+00,
#        1.17388466e+00, 2.64754518e+00, 6.16041685e-02, 2.28682342e-04,
#        3.23060165e+00, 2.28493891e+00, 1.31586772e+00, 3.80322320e-02,
#        1.93071950e+00, 8.96887895e-01, 0.00000000e+00, 8.92466612e-01,
#        1.90029075e+00, 1.13165850e+00, 2.20892624e+00, 2.29768719e+00,
#        1.23402055e+00, 1.99159520e+00, 2.79704534e+00, 1.41340143e+00,
#        2.25534517e+00, 4.42059248e+00, 1.67442702e+00, 9.44127611e-01,
#        4.70777062e+00, 1.34640698e+00, 3.34232794e+00, 3.41015998e+00]

backend = AerSimulator(method="statevector")
def calculate_cost_function(thetas):

    #update the variational parameters
    qc_anz0 = qc_anz.assign_parameters(thetas)
    qc_anz0.save_statevector()
    sv_anz = backend.run(qc_anz0).result().get_statevector().data

    denom = np.vdot(sv_anz, obs_beta @ sv_anz)
    numer = np.vdot(sv_anz, obs_delta @ sv_anz)

    cost = 0.5*(1 - 1/(nqubit-1) * numer.real/denom.real)
    return(cost)

#callback to store the cost function at each iteration
cost_history = []
x_history = []
def store_intermediate_result(xk):
    x_history.append(xk.copy())
    val = calculate_cost_function(xk)
    cost_history.append(val)
    print(val)

#Optimizer parameters
nitr = 500
maxfun = 1e6
tol = 1e-6
gtol = 1e-6
eps = 1e-8
bounds = [(0, 2 * np.pi)] * params['ntheta']

if params['opt']=='CG':
    optimizer = CG(maxiter=nitr, tol=tol, gtol=gtol, callback=store_intermediate_result)
    theta_opt = optimizer.minimize(fun=calculate_cost_function, x0=thetas)
elif params['opt']=='L_BFGS_B':
    optimizer = L_BFGS_B(maxfun=maxfun,maxiter=nitr, eps=eps, ftol=tol, callback=store_intermediate_result)
    theta_opt = optimizer.minimize(fun=calculate_cost_function,x0=thetas,bounds=bounds)
elif params['opt']=='GradientDescent':
    optimizer = GradientDescent(maxiter=nitr, tol=tol, callback=store_intermediate_result)
    theta_opt = optimizer.minimize(fun=calculate_cost_function, x0=thetas)


print(theta_opt)


In [ ]:
#Plot the cost function history
plt.plot(cost_history, marker='o')
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Cost function per iteration")
plt.yscale('log')
plt.show()


In [ ]:

#Evaluate ansatz with optimized parameters
qc_anz0 = qc_anz.assign_parameters(theta_opt.x)
qc_anz0.save_statevector()
backend = AerSimulator(method="statevector")
job = backend.run(qc_anz0)
result = job.result()
sv_anz = result.get_statevector().data.real

#Extract quantum solution at each time step
x_qutm = np.zeros(nt*nx)
delta0 = np.sum([int(nx**j) for j in range(1,alpha+1)])
for i in range(nt):
    offset = int((i+1)*2*nx**alpha - delta0)
    x_qutm[i*nx:(i+1)*nx] = sv_anz[offset:nx+offset]
#print(np.sqrt(np.mean((x_qutm-x_real)**2)))

#Classical solution
Le_real = Carleman_Dilation.Carleman_Dilation_Matrix(params)
x_real = mu.solve_sys(Le_real,Be,params)

#Plot real solution with quantum solution
fig, ax = plt.subplots(1,1)
p1 = ax.plot(x_qutm)
p2 = ax.plot(x_real)
plt.savefig("Solution.png")


In [ ]:
#Either create and transpile circuits or load from file
#Note: Transpiling takes several minutes
if (lCreateCircs):
    #Save beta_ij and delta_ijk circuits to files
    circs_beta = []
    circs_delta = []
    for i in range(0,Ns):
        for j in range(0,Ns):
            circs_beta.append(cu.create_beta_ij(i,j,qc_carl,qc_anz,params))
            for k in range(0,nqubit-1):
                circs_delta.append(cu.create_delta_ijk(i,j,k,qc_carl,qc_anz,qc_init,params))

    #Transpile all circuits
    pass_manager = generate_preset_pass_manager(optimization_level=3, backend=AerSimulator())
    print('transpiling beta circuits')
    circs_beta = pass_manager.run(circs_beta)
    print('transpiling delta circuits')
    circs_delta = pass_manager.run(circs_delta)

    #Save circuits to file
    print('Saving beta circuits')
    with open("beta_circuits.qpy", "wb") as f:
        qpy.dump(circs_beta, f)

    #Save circuits to file
    print('Saving delta circuits')
    with open("delta_circuits.qpy", "wb") as f:
        qpy.dump(circs_delta, f)

else:
    #Load from file and store circuits in cache
    with open("beta_circuits.qpy", "rb") as f:
        circs_beta = qpy.load(f)
    with open("delta_circuits.qpy", "rb") as f:
        circs_delta = qpy.load(f)


In [ ]:
# #Cost function
# #FLAG: I think certain configurations are always zero. Check this to reduce cost
# def calculate_cost_function(thetas):
#     denom = cu.measure_beta_ij(circs_beta,coeffs_ij,thetas)
#     numer = cu.measure_delta_ijk(circs_delta,coeffs_ijk,thetas)
#     cost = 0.5*(1 - 1/(nqubit-1) * numer/denom)
#     print(cost)
#     return(cost)


# #cost = calculate_cost_function(thetas)


# #Optimizer
# nitr = 1000
# tol = 1e-2
# gtol = 1e-2
# # optimizer = CG(maxiter=nitr, tol=tol, gtol=gtol) #, callback=store_intermediate_result_1)
# # result = optimizer.minimize(fun=calculate_cost_function, x0=thetas)

# optimizer = GradientDescent(maxiter=nitr, tol=tol) #callback=store_intermediate_result_4)
# result = optimizer.minimize(fun=calculate_cost_function, x0=thetas)
# print(result)
